# Análise de Sentimento em Comentários de Redes Sociais

Projeto de ponta a ponta: carregamento de um dataset público de posts de redes sociais (Twitter, Instagram, Facebook) com rótulos de sentimento, análise exploratória, limpeza de texto, engenharia de features, divisão treino/teste sem leakage, treino e avaliação de um classificador, e sete perguntas de negócio respondidas com os dados.

In [ ]:
# ==============================================================================
# SETUP KAGGLE: SparkSession + display()
# ==============================================================================
# No Databricks, `spark` e `display()` vêm incluídos. No Kaggle/Jupyter é preciso
# criá-los — esta célula torna o notebook executável nas duas plataformas.
# ==============================================================================

import importlib.util, subprocess, sys

if importlib.util.find_spec("pyspark") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyspark"])

from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("analise-sentimento").getOrCreate()

from IPython.display import display as _display
def display(x, **kwargs):
    """display() do Databricks, adaptado: Spark DF -> pandas -> IPython display."""
    if hasattr(x, "toPandas"):
        _display(x.toPandas())
    else:
        _display(x, **kwargs)

print("Setup concluído. Spark:", spark.version)


In [ ]:
# ==============================================================================
# STEP 1: CARREGAMENTO DO DATASET
# ==============================================================================
# Objectivo: carregar o dataset público de sentimento em redes sociais
#            num Spark DataFrame.
#
# Input:  CSV anexado no Kaggle (/kaggle/input) OU parquet remoto (Hugging Face)
# Output: df_raw (Spark DataFrame)
#
# Operações:
#   - se existir um CSV anexado no Kaggle (Add Input), ler de lá
#     (funciona mesmo com a Internet desligada)
#   - caso contrário, obter o parquet via API do Hugging Face
#   - normalizar nomes de colunas e valores com espaços a mais
# ==============================================================================

import json, glob, urllib.request
import pandas as pd

# Dataset público: posts de Twitter/Instagram/Facebook com rótulo de sentimento
DS_ID = "chuuhtetnaing/myanmar-social-media-sentiment-analysis-dataset"

# 1) Procurar CSV anexado no Kaggle (painel "Add Input" -> /kaggle/input/...)
csvs = sorted(glob.glob("/kaggle/input/**/*.csv", recursive=True))
if csvs:
    print("CSV anexado encontrado:", csvs[0])
    pdf = pd.read_csv(csvs[0])
    ORIGEM = "CSV anexado (Kaggle Dataset)"
else:
    # 2) Fallback: descarregar o parquet via API do Hugging Face
    with urllib.request.urlopen(
        f"https://datasets-server.huggingface.co/parquet?dataset={DS_ID}", timeout=30
    ) as r:
        urls = [f["url"] for f in json.load(r)["parquet_files"]]
    pdf = pd.concat([pd.read_parquet(u) for u in urls], ignore_index=True)
    ORIGEM = "Hugging Face (parquet remoto)"
print("Origem dos dados:", ORIGEM)

# 3) Normalizar colunas e valores textuais (vêm com espaços a mais)
pdf.columns = [c.strip() for c in pdf.columns]
for col in ["Text", "Sentiment", "Platform", "Hashtags", "Country"]:
    pdf[col] = pdf[col].astype(str).str.strip()

# 4) Converter para Spark DataFrame
df_raw = spark.createDataFrame(pdf)

print("Linhas:", df_raw.count(), "| Colunas:", len(df_raw.columns))
display(df_raw.limit(5))


In [ ]:
# ==============================================================================
# STEP 2: ANÁLISE EXPLORATÓRIA (EDA)
# ==============================================================================
# Objectivo: conhecer a distribuição dos dados antes de qualquer limpeza
#            ou modelação.
#
# Input:  df_raw (Spark DataFrame)
# Output: contagens e distribuições (Sentiment, Platform, nulos, comprimento)
#
# Operações:
#   - distribuição da variável alvo (Sentiment)
#   - contagem por plataforma e por país
#   - verificação de nulos e duplicados
#   - estatísticas do comprimento do texto
# ==============================================================================

from pyspark.sql import functions as F

# 1) Distribuição da variável alvo
df_raw.groupBy("Sentiment").count().orderBy(F.desc("count")).show()

# 2) Contagens por plataforma e por país
df_raw.groupBy("Platform").count().orderBy(F.desc("count")).show()
df_raw.groupBy("Country").count().orderBy(F.desc("count")).show()

# 3) Qualidade: nulos e duplicados
print("Nulos por coluna:")
df_raw.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df_raw.columns]).show()
print("Duplicados exatos (texto):", df_raw.count() - df_raw.dropDuplicates(["Text"]).count())

# 4) Comprimento do texto por classe de sentimento
df_raw = df_raw.withColumn("text_len", F.length("Text"))
df_raw.groupBy("Sentiment").agg(
    F.round(F.avg("text_len"), 1).alias("len_medio"),
    F.min("text_len").alias("len_min"),
    F.max("text_len").alias("len_max"),
    F.count("*").alias("n")
).show()

In [ ]:
# ==============================================================================
# STEP 3: CONSOLIDAÇÃO DOS RÓTULOS EM 3 CLASSES + DEDUP
# ==============================================================================
# Objectivo: reduzir as 191 emoções do dataset para 3 classes treino/avaliáveis
#            (Positivo / Negativo / Neutro) e remover textos duplicados.
#
# Input:  df_raw (coluna Sentiment com 191 rótulos)
# Output: df (coluna label com 3 classes, sem duplicados)
#
# Operações:
#   - classificar cada rótulo por palavras-chave
#   - remover duplicados (evita leakage treino/teste)
#   - verificar distribuição final
# ==============================================================================

from pyspark.sql import functions as F

# Palavras-chave verificadas contra os 191 rótulos existentes
NEG_KW = ["negativ", "sad", "despair", "grief", "lonel", "anger", "angry", "fear",
          "anxi", "disgust", "frustrat", "regret", "bitter", "betray", "heartbreak",
          "heartach", "sorrow", "loss", "lost", "shame", "jealous", "envy", "resent",
          "devast", "desolat", "helpless", "isolat", "disappoint", "suffer", "desperat",
          "darkness", "exhaust", "overwhelm", "melancholy", "apprehens", "intimidat",
          "dismiss", "hate", "bad", "storm", "ruins", "obstacle", "pressure", "miscalculat"]
NEU_KW = ["neutral", "curios", "confus", "indiffer", "ambival", "nostalgia", "bittersweet",
          "reflect", "contemplat", "pensive", "intrigue", "suspense", "boredom", "numb",
          "solitude", "challenge"]

# Classificar: neutro primeiro (p.ex. "bittersweet" não pode cair em "bitter")
def classificar(rotulo):
    r = str(rotulo).lower()
    if any(k in r for k in NEU_KW):
        return "Neutro"
    if any(k in r for k in NEG_KW):
        return "Negativo"
    return "Positivo"

# Aplicar em pandas (732 linhas) e voltar a Spark
pdf = df_raw.toPandas()
pdf["label"] = pdf["Sentiment"].map(classificar)

# Remover duplicados ANTES do split treino/teste
n_antes = len(pdf)
pdf = pdf.drop_duplicates(subset=["Text"]).reset_index(drop=True)
print(f"Linhas: {n_antes} -> {len(pdf)} (removidos {n_antes - len(pdf)} duplicados)")

df = spark.createDataFrame(pdf)
df.groupBy("label").count().orderBy(F.desc("count")).show()

In [ ]:
# ==============================================================================
# INSPEÇÃO: ESTRUTURA DOS DATASETS (PRIMEIRAS 51 LINHAS)
# ==============================================================================
# Objectivo: mostrar as primeiras 51 linhas de cada dataset com TODAS as
#            colunas, para perceber a estrutura dos dados.
#
# Input:  df_raw (original, 732 linhas) e df (processado, 706 linhas)
# Output: duas tabelas de 51 linhas, uma por dataset
# ==============================================================================

print("=" * 70)
print(f"DATASET ORIGINAL (df_raw): {df_raw.count()} linhas x {len(df_raw.columns)} colunas")
print("Colunas:", df_raw.columns)
display(df_raw.limit(51))

print("=" * 70)
print(f"DATASET PROCESSADO (df): {df.count()} linhas x {len(df.columns)} colunas")
print("Colunas:", df.columns)
display(df.limit(51))

In [ ]:
# ==============================================================================
# STEP 4: LIMPEZA DE TEXTO
# ==============================================================================
# Objectivo: criar uma coluna clean_text pronta para vectorização, sem ruído,
#            preservando o Text original para as perguntas de negócio.
#
# Input:  df (coluna Text)
# Output: colunas clean_text e n_emojis
#
# Operações:
#   - contar emojis antes de os remover (pergunta P5 precisa deles)
#   - minúsculas, remover URLs, hashtags, emojis e não-letras
#   - normalizar espaços e verificar textos vazios
# ==============================================================================

from pyspark.sql import functions as F

# Intervalos Unicode de emojis/pictogramas
RE_EMOJI = "[\U0001F000-\U0001FAFF\u2600-\u27BF\uFE0F\u200D]"

# 1) Contar emojis ANTES de os remover
df = df.withColumn(
    "n_emojis",
    F.length("Text") - F.length(F.regexp_replace("Text", RE_EMOJI, ""))
)

# 2) Limpeza progressiva (a ordem importa)
df = (
    df.withColumn("clean_text", F.lower("Text"))
      .withColumn("clean_text", F.regexp_replace("clean_text", "http\\S+|www\\.\\S+", " "))  # URLs
      .withColumn("clean_text", F.regexp_replace("clean_text", "#\\w+", " "))               # hashtags (já têm coluna própria)
      .withColumn("clean_text", F.regexp_replace("clean_text", RE_EMOJI, " "))              # emojis
      .withColumn("clean_text", F.regexp_replace("clean_text", "[^a-z\\s]", " "))           # só letras
      .withColumn("clean_text", F.trim(F.regexp_replace("clean_text", "\\s+", " ")))       # espaços
)

# 3) Verificar resultado
display(df.select("Text", "clean_text", "n_emojis").limit(5))
print("Textos vazios após limpeza:", df.filter(F.col("clean_text") == "").count())

In [ ]:
# ==============================================================================
# STEP 5: ENGENHARIA DE FEATURES
# ==============================================================================
# Objectivo: criar features numéricas simples, determinísticas, por linha.
#
# Input:  df (colunas clean_text, Hashtags, Text)
# Output: n_words, n_hashtags, n_exclam, n_question
#
# Operações:
#   - nº de palavras do texto limpo
#   - nº de hashtags (coluna própria)
#   - nº de "!" e "?" no texto original
#   - médias por classe (ver sinal preditivo)
#
# Nota anti-leakage: o TF-IDF só será ajustado DEPOIS do split, no treino,
#                     porque aprende o vocabulário a partir dos dados.
# ==============================================================================

from pyspark.sql import functions as F

# 1) Features por linha (determinísticas, sem aprender dos dados)
df = (
    df.withColumn("n_words", F.size(F.split(F.col("clean_text"), " ")))
      .withColumn("n_hashtags", F.when(
              F.col("Hashtags").isNull()
              | (F.trim("Hashtags") == "")
              | (F.lower("Hashtags") == "nan"), 0
          ).otherwise(F.size(F.split(F.trim("Hashtags"), "\\s+"))))
      .withColumn("n_exclam", F.length("Text") - F.length(F.regexp_replace("Text", "!", "")))
      .withColumn("n_question", F.length("Text") - F.length(F.regexp_replace("Text", "\\?", "")))
)

# 2) Verificar sinal: médias das features por classe
FEATURES = ["n_words", "n_hashtags", "n_exclam", "n_question", "n_emojis", "text_len"]
df.groupBy("label").agg(*[F.round(F.avg(c), 2).alias(c) for c in FEATURES]).show()

display(df.select("Text", "n_words", "n_hashtags", "n_exclam", "n_question").limit(5))

In [ ]:
# ==============================================================================
# STEP 6: DIVISÃO TREINO/TESTE (ANTES DA VETORIZAÇÃO)
# ==============================================================================
# Objectivo: separar 20% dos dados para teste, de forma estratificada,
#            garantindo zero leakage no que se segue.
#
# Input:  df (706 linhas, coluna label)
# Output: train / test (pandas, ~80% / ~20%)
#
# Operações:
#   - extrair colunas de modelação para pandas
#   - split estratificado por label com seed fixa
#   - confirmar proporções idênticas nos dois conjuntos
# ==============================================================================

from sklearn.model_selection import train_test_split

# 1) Colunas necessárias à modelação
data = df.select(
    "clean_text", "label",
    "n_words", "n_hashtags", "n_exclam", "n_question", "n_emojis", "text_len"
).toPandas()

# 2) Split estratificado (seed fixa = reprodutível)
train, test = train_test_split(
    data, test_size=0.2, stratify=data["label"], random_state=42
)
print(f"Treino: {len(train)} linhas | Teste: {len(test)} linhas")

# 3) Confirmar: proporções de classes iguais nos dois conjuntos
comp = pd.concat([
    train["label"].value_counts(normalize=True).rename("Treino"),
    test["label"].value_counts(normalize=True).rename("Teste"),
], axis=1)
display(comp)

In [ ]:
# ==============================================================================
# STEP 7: TREINO DO CLASSIFICADOR
# ==============================================================================
# Objectivo: treinar um pipeline TF-IDF + Regressão Logística, ajustado
#            EXCLUSIVAMENTE ao conjunto de treino.
#
# Input:  train (564 linhas, clean_text + label)
# Output: modelo ajustado + F1 macro em validação cruzada
#
# Operações:
#   - pipeline TF-IDF (uni+bigramas) -> Regressão Logística
#   - validação cruzada 5-fold SÓ no treino (o teste fica intocado)
#   - ajuste final ao treino completo
# ==============================================================================

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

# Pipeline: o TF-IDF é ajustado dentro, só com dados de treino
modelo = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced")),
])

# Validação cruzada 5-fold (dentro do treino) — estimativa honesta antes do teste
f1_cv = cross_val_score(modelo, train["clean_text"], train["label"], cv=5, scoring="f1_macro")
print("F1 macro por fold:", f1_cv.round(3), "| média:", round(f1_cv.mean(), 3))

# Ajuste final ao treino completo
modelo.fit(train["clean_text"], train["label"])
print("Termos no vocabulário:", len(modelo.named_steps["tfidf"].vocabulary_))

In [ ]:
# ==============================================================================
# STEP 8: AVALIAÇÃO NO TESTE
# ==============================================================================
# Objectivo: medir o desempenho real do modelo no conjunto de teste
#            (nunca visto pelo treino nem pela vetorização).
#
# Input:  modelo + test (142 linhas)
# Output: acurácia, F1 macro, relatório por classe, matriz de confusão
#
# Operações:
#   - prever no teste
#   - acurácia e F1 macro
#   - relatório por classe (precisão/recall/F1)
#   - matriz de confusão
# ==============================================================================

from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

# 1) Previsões no teste (primeiro contacto do modelo com estes dados)
pred = modelo.predict(test["clean_text"])
y_true = test["label"]

# 2) Métricas globais
print("Acurácia:", round(accuracy_score(y_true, pred), 3))
print("F1 macro:", round(f1_score(y_true, pred, average="macro"), 3))
print()
print(classification_report(y_true, pred))

# 3) Matriz de confusão (linhas = real, colunas = previsto)
cm = confusion_matrix(y_true, pred, labels=modelo.classes_)
display(pd.DataFrame(cm, index=modelo.classes_, columns=modelo.classes_))

In [ ]:
# ==============================================================================
# P1: QUAL A PLATAFORMA COM MAIS SENTIMENTO NEGATIVO?
# ==============================================================================
# Pergunta: qual das plataformas (Twitter, Instagram, Facebook) tem a maior
#            proporção de comentarios negativos?
#
# Justificacao: saber onde vive o sentimento negativo permite priorizar
#               esforcos de moderacao e resposta ao cliente por plataforma.
#
# Resposta: ver output - proporcao de Negativo por plataforma (rotulos reais).
# ==============================================================================

from pyspark.sql import functions as F

# Proporcao de negativos por plataforma (usando rotulos reais, n=706)
p1 = (df.groupBy("Platform")
        .agg(F.count("*").alias("total"),
             F.sum(F.when(F.col("label") == "Negativo", 1).otherwise(0)).alias("negativos"))
        .withColumn("pct_negativo", F.round(100 * F.col("negativos") / F.col("total"), 1))
        .orderBy(F.desc("pct_negativo")))
display(p1)

In [ ]:
# ==============================================================================
# P2: QUE HORÁRIOS GERAM MAIS COMENTÁRIOS TÓXICOS?
# ==============================================================================
# Pergunta: em que horas do dia a proporção de comentários tóxicos
#            (aqui: sentimento Negativo) é mais alta?
#
# Justificação: dimensionar equipas de moderação para os horários de risco
#               e detetar padrões de comportamento por hora do dia.
#
# Resposta: ver output — % Negativo por hora (coluna Hour, rótulos reais).
# ==============================================================================

from pyspark.sql import functions as F

# Nota: "tóxico" é aproximação honesta — o dataset não tem coluna de toxicidade;
# usamos a classe Negativo como proxy mais próximo.

# 1) Negativos e % por hora do dia
p2 = (df.groupBy("Hour")
        .agg(F.count("*").alias("total"),
             F.sum(F.when(F.col("label") == "Negativo", 1).otherwise(0)).alias("negativos"))
        .withColumn("pct_negativo", F.round(100 * F.col("negativos") / F.col("total"), 1)))
display(p2.orderBy("Hour"))

# 2) Top 3 horas mais tóxicas (com mínimo de amostra para evitar ruído)
top3 = p2.filter(F.col("total") >= 20).orderBy(F.desc("pct_negativo")).limit(3)
print("Horas com maior % de negativos (>=20 posts):")
display(top3)

In [ ]:
# ==============================================================================
# P3: HÁ PICOS DE DESINFORMAÇÃO EM DATAS ESPECÍFICAS?
# ==============================================================================
# Pergunta: existem datas com picos anormais de atividade negativa,
#            padrão tipicamente associado a vagas de desinformação?
#
# Justificação: o dataset NÃO tem rótulo de desinformação (verificado).
#               Usamos proxy: rajadas de posts negativos concentrados
#               numa mesma data. Campanhas de desinformação geram
#               exatamente esse padrão de pico.
#
# Resposta: ver output — datas com mais negativos (absoluto e %).
# ==============================================================================

from pyspark.sql import functions as F

# 0) Confirmação: procurar marcas diretas de desinformação (palavras inteiras)
kw = r"\b(fake|misinformation|disinformation|rumor|rumour|hoax|conspiracy|scam|untrue|false)\b"
diretos = df.filter(F.lower("Text").rlike(kw) | F.lower("Hashtags").rlike(kw)).count()
print("Posts com termos diretos de desinformação:", diretos)

# 1) Negativos por data (Timestamp -> data)
p3 = (df.withColumn("data", F.to_date("Timestamp"))
        .groupBy("data")
        .agg(F.count("*").alias("total"),
             F.sum(F.when(F.col("label") == "Negativo", 1).otherwise(0)).alias("negativos"))
        .withColumn("pct_negativo", F.round(100 * F.col("negativos") / F.col("total"), 1)))

# 2) Datas com mais negativos em absoluto
display(p3.orderBy(F.desc("negativos")).limit(5))

# 3) Datas com maior concentração de negativos (mínimo 10 posts)
display(p3.filter(F.col("total") >= 10).orderBy(F.desc("pct_negativo")).limit(5))

In [ ]:
# ==============================================================================
# P4: QUAIS AS PALAVRAS MAIS ASSOCIADAS A SENTIMENTO POSITIVO?
# ==============================================================================
# Pergunta: que palavras ocorrem desproporcionalmente nos posts positivos?
#
# Justificação: identificar vocabulário positivo apoia brand monitoring,
#               respostas automáticas e calibração de comunicação.
#
# Resposta: ver output — top 15 palavras por contagem + "lift"
#           (quanto mais frequente nos positivos do que no geral).
# ==============================================================================

from collections import Counter
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from pyspark.sql import functions as F

# 1) Textos positivos e todos os textos, para pandas
pos = df.filter(F.col("label") == "Positivo").select("clean_text").toPandas()
todos = df.select("clean_text").toPandas()

# 2) Contar palavras (sem stopwords, com 3+ letras)
def contar(series):
    cnt = Counter()
    for t in series:
        for w in str(t).split():
            if w not in ENGLISH_STOP_WORDS and len(w) > 2:
                cnt[w] += 1
    return cnt

cnt_pos, cnt_todos = contar(pos["clean_text"]), contar(todos["clean_text"])

# 3) Top 15 + lift: presença nos positivos vs presença global
top = pd.DataFrame(cnt_pos.most_common(15), columns=["palavra", "n"])
top["pct_posts_positivos"] = (100 * top["n"] / len(pos)).round(1)
top["pct_todos"] = top["palavra"].map(lambda w: 100 * cnt_todos.get(w, 0) / len(todos)).round(1)
top["lift"] = (top["pct_posts_positivos"] / top["pct_todos"]).round(2)
display(top)

In [ ]:
# ==============================================================================
# P5: QUAL A PERCENTAGEM DE COMENTÁRIOS SÓ COM EMOJIS E QUAL O SENTIMENTO?
# ==============================================================================
# Pergunta: que proporção dos comentários é exclusivamente emojis (sem texto)?
#            E que sentimento têm os posts que usam emojis?
#
# Justificação: mensagens só com emojis desafiam moderação e análise de
#               sentimento tradicionais; medir o seu peso e o seu viés
#               emocional informa o desenho do pipeline.
#
# Resposta: ver output — % de posts só-emoji + sentimento dos posts com emoji.
# ==============================================================================

from pyspark.sql import functions as F

total = df.count()

# 1) Posts só com emojis: têm emojis e nenhum texto restante após limpeza
so_emoji = df.filter((F.col("n_emojis") > 0) & (F.col("clean_text") == ""))
print(f"Posts só com emojis: {so_emoji.count()} de {total} "
      f"({100 * so_emoji.count() / total:.2f}%)")

# 2) Posts que contêm emojis (mesmo acompanhados de texto)
com_emoji = df.filter(F.col("n_emojis") > 0)
print(f"Posts com pelo menos 1 emoji: {com_emoji.count()} de {total} "
      f"({100 * com_emoji.count() / total:.2f}%)")

# 3) Sentimento dos posts com emojis vs global
resumo = (
    com_emoji.groupBy("label").agg(F.count("*").alias("n"))
            .withColumn("pct", F.round(100 * F.col("n") / com_emoji.count(), 1))
            .withColumn("grupo", F.lit("com_emoji"))
)
global_resumo = (
    df.groupBy("label").agg(F.count("*").alias("n"))
      .withColumn("pct", F.round(100 * F.col("n") / total, 1))
      .withColumn("grupo", F.lit("todos"))
)
display(resumo.unionByName(global_resumo).orderBy("grupo", F.desc("n")))

In [ ]:
# ==============================================================================
# P6: OS COMENTÁRIOS MAIS LONGOS TENDEM A SER MAIS NEGATIVOS?
# ==============================================================================
# Pergunta: existe relação entre o comprimento do comentário e a
#            probabilidade de ser negativo?
#
# Justificação: crença comum em gestão de comunidades é que textos longos
#               = queixas. Confirmar ou refutar orienta respostas automáticas
#               e regras de triagem por comprimento.
#
# Resposta: ver output — comprimento médio por classe, % negativo por faixa
#           de comprimento (quartis) e correlação.
# ==============================================================================

from pyspark.sql import functions as F

# 1) Comprimento médio por classe
display(df.groupBy("label").agg(
    F.count("*").alias("n"),
    F.round(F.avg("text_len"), 1).alias("text_len_medio"),
    F.round(F.avg("n_words"), 1).alias("n_words_medio")
).orderBy("label"))

# 2) Faixas de comprimento (quartis) e % negativo em cada uma
q1, q2, q3 = df.approxQuantile("text_len", [0.25, 0.5, 0.75], 0)
print(f"Quartis de text_len: {q1} / {q2} / {q3}")

df6 = df.withColumn("faixa",
    F.when(F.col("text_len") <= q1, "1. curto (Q1)")
     .when(F.col("text_len") <= q2, "2. medio (Q2)")
     .when(F.col("text_len") <= q3, "3. longo (Q3)")
     .otherwise("4. muito longo (Q4)"))

p6 = (df6.groupBy("faixa")
          .agg(F.count("*").alias("total"),
               F.sum(F.when(F.col("label") == "Negativo", 1).otherwise(0)).alias("negativos"),
               F.round(F.avg("text_len"), 1).alias("len_medio"))
          .withColumn("pct_negativo", F.round(100 * F.col("negativos") / F.col("total"), 1)))
display(p6.orderBy("faixa"))

# 3) Correlação comprimento <-> negatividade (Pearson)
corr = df.withColumn("neg", (F.col("label") == "Negativo").cast("int")) \
         .agg(F.round(F.corr("text_len", "neg"), 3).alias("corr_text_len_negativo"))
display(corr)

In [ ]:
# ==============================================================================
# P7: QUAL O TÓPICO OU HASHTAG COM MAIOR VOLUME DE CONVERSA?
# ==============================================================================
# Pergunta: que hashtags geram mais posts? E que sentimento dominam?
#
# Justificação: identificar os tópicos com maior volume orienta a estratégia
#               de conteúdo, campanhas e monitorização de tendências.
#
# Resposta: ver output — top 15 hashtags por nº de posts + sentimento do top 5.
# ==============================================================================

from pyspark.sql import functions as F

# 1) Explodir a coluna Hashtags (cada hashtag vira uma linha)
hashtags = (df.withColumn("hashtag", F.explode(F.split(F.trim("Hashtags"), "\\s+")))
              .filter(F.col("hashtag").rlike("^#\\w+"))
              .filter((F.lower("hashtag") != "#nan") & (F.col("hashtag") != "nan")))

print("Hashtags distintas:", hashtags.select("hashtag").distinct().count())

# 2) Top 15 por volume de posts
top_hashtags = (hashtags.groupBy("hashtag")
                        .agg(F.count("*").alias("posts"))
                        .orderBy(F.desc("posts"))
                        .limit(15))
display(top_hashtags)

# 3) Sentimento dos 5 tópicos mais falados
top5 = [r["hashtag"] for r in top_hashtags.limit(5).collect()]
resumo = (hashtags.filter(F.col("hashtag").isin(top5))
                  .groupBy("hashtag", "label")
                  .agg(F.count("*").alias("n")))
display(resumo.orderBy("hashtag", F.desc("n")))

# Conclusões

## Pipeline
Dataset público de 732 posts de redes sociais (Twitter, Instagram, Facebook) com rótulos de sentimento, reduzido a **706 linhas sem duplicados** após consolidação das 191 emoções originais em 3 classes (Positivo 65% / Negativo 23% / Neutro 12%). Classificador TF-IDF + Regressão Logística, treinado sem leakage (split estratificado 80/20 antes da vetorização):

* **Acurácia no teste: 0.81 | F1 macro: 0.68**
* Negativo F1 0.83, Positivo F1 0.87, Neutro F1 0.34 (classe difícil, confundida sobretudo com Positivo)

## Respostas às 7 perguntas de negócio

| # | Pergunta | Resposta |
| --- | --- | --- |
| P1 | Plataforma com mais sentimento negativo | **Twitter (24.3%)**, seguida de Facebook (22.9%) e Instagram (21.4%) — diferença pequena |
| P2 | Horários com mais comentários tóxicos | **9h (37%), 21h (32.5%) e 14h (27.5%)** (proxy: sentimento Negativo) |
| P3 | Picos de desinformação em datas específicas | **Não identificável** — sem marcadores de desinformação e sem densidade temporal suficiente (nenhuma data tem ≥10 posts) |
| P4 | Palavras mais associadas a sentimento positivo | **friends (1.57×), beauty e dance (1.46×), joy (1.45×), nature (1.44×), concert (1.43×)** — por lift vs. presença global |
| P5 | % de comentários só com emojis e o seu sentimento | **0%** só emojis; 11 posts (1.56%) contêm emojis e **todos são Positivos** |
| P6 | Comentários mais longos tendem a ser mais negativos? | **Não** — Negativos têm a média mais curta (80.4 chars); correlação comprimento↔negativo ≈ −0.05 (nula) |
| P7 | Tópico/hashtag com maior volume | **#Serenity (15 posts)** num leque muito fragmentado de 975 hashtags; #Despair é o tópico negativo com mais volume (11) |

## Limitações

* Dataset sintético e pequeno (706 linhas): percentagens em cortes finos (horas, datas, hashtags) têm amostras reduzidas.
* "Tóxico" (P2) é aproximado por sentimento Negativo — o dataset não tem rótulo de toxicidade.
* A desinformação (P3) não é mensurável com estes dados.
* Os hashtags espelham os rótulos de emoção do dataset, pelo que P7 reflete mais a estrutura do dataset do que conversa orgânica.

In [ ]:
# ==============================================================================
# GRÁFICOS: P1, P2 E P7 (perguntas comparativas)
# ==============================================================================
# Objectivo: mostrar numa só figura os três gráficos das perguntas de
#            negócio comparativas (plataforma, hora, hashtag).
#
# Input:  p1, p2, top_hashtags (calculados nas células P1, P2 e P7)
# Output: figura matplotlib com 3 painéis
# ==============================================================================

import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# P1 — % negativo por plataforma
p1pd = p1.toPandas()
axes[0].bar(p1pd["Platform"], p1pd["pct_negativo"], color="#c0392b")
axes[0].set_title("P1 — % Negativo por Plataforma")
axes[0].set_ylabel("% negativo")
axes[0].set_ylim(0, 30)

# P2 — % negativo por hora do dia
p2pd = p2.orderBy("Hour").toPandas()
axes[1].bar(p2pd["Hour"], p2pd["pct_negativo"], color="#e67e22")
axes[1].set_title("P2 — % Negativo por Hora")
axes[1].set_xlabel("hora do dia")
axes[1].set_ylabel("% negativo")

# P7 — top 15 hashtags por volume
th = top_hashtags.toPandas()
axes[2].barh(th["hashtag"][::-1], th["posts"][::-1], color="#2980b9")
axes[2].set_title("P7 — Top 15 Hashtags")
axes[2].set_xlabel("nº de posts")

plt.tight_layout()
display(fig)